In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Load the data
brussels_residents = pd.read_csv("../7_activity_times_modes/output/workers_with_activity.csv")
brussels_commuters = pd.read_csv("../8_brussels_commuters/output/commuters_travelling.csv")

In [ ]:
def compute_duration_minutes(start_time, end_time):
    """Compute duration in minutes, allowing overnight wrap-around."""
    start_dt = pd.to_datetime(start_time, errors="coerce")
    end_dt = pd.to_datetime(end_time, errors="coerce")
    duration = (end_dt - start_dt).dt.total_seconds() / 60
    duration = duration.where(duration >= 0, duration + 24 * 60)
    return duration

# Map residents to final schema
res = brussels_residents.copy()

res_mapped = pd.DataFrame({
    "sex": res["sex"],
    "age": res["age"],
    "education": res["education"],
    "industry": res["industry"],
    "professional_status": res["professional_status"],
    "home_municipality": res["assigned_sector"].astype(str).str[:5],
    "home_x": res["home_x"],
    "home_y": res["home_y"],
    "assigned_work_sector": res["work_sector"],
    "work_x": res["work_x"],
    "work_y": res["work_y"],
    "has_car": res["has_car"],
    "has_company_car": res.get("has_company_car", np.nan),
    "commute_dist_km": pd.to_numeric(res["commute_dist_km"], errors="coerce"),
    "departure_home_to_work": res["departure_home_to_work"],
    "arrival_work": res["arrival_work"],
    "departure_from_work": res["departure_from_work"],
    "work_duration": res["work_duration"],
    "assigned_mode": res["assigned_mode"],
    "hts_source_id": res.get("hts_source_id", np.nan),
    "lives_in_brussels": True,
    "works_in_brussels": res["work_sector"].astype(str).str.startswith("210"),
    "population_type": "resident"
})

# Map commuters to final schema
com = brussels_commuters.copy()

com_mapped = pd.DataFrame({
    "sex": com["sex"],
    "age": com["age_group"],
    "education": com["education"],
    "industry": com["profession"],
    "professional_status": com["profession"],
    "home_municipality": com["municipality_nis"],
    "home_x": com["home_x"],
    "home_y": com["home_y"],
    "assigned_work_sector": com["assigned_work_sector"],
    "work_x": com["work_x"],
    "work_y": com["work_y"],
    "has_car": com["has_car"],
    "has_company_car": com["has_company_car"],
    "commute_dist_km": pd.to_numeric(com["commute_dist_km"], errors="coerce"),
    "departure_home_to_work": com["departure_home_to_work"],
    "arrival_work": com["arrival_work"],
    "departure_from_work": com["departure_from_work"],
    "assigned_mode": com["mode"],
    "hts_source_id": com["participant_id"],
    "lives_in_brussels": False,
    "works_in_brussels": True,
    "population_type": "commuter"
})
com_mapped["work_duration"] = com_mapped["departure_from_work"] - com_mapped["arrival_work"]

# Merge and save
final_columns = [
    "sex", "age", "education", "industry", "professional_status",
    "home_municipality", "home_x", "home_y",
    "assigned_work_sector", "work_x", "work_y", "has_car", "has_company_car",
    "commute_dist_km", "departure_home_to_work", "arrival_work", "work_duration",
    "departure_from_work", "assigned_mode", "hts_source_id",
    "lives_in_brussels", "works_in_brussels", "population_type"
]

workers_merged = pd.concat([res_mapped[final_columns], com_mapped[final_columns]], ignore_index=True)
workers_merged.to_csv("output/workers_merged.csv", index=False)

print("Saved: output/workers_merged.csv")

In [ ]:
print(f"Residents mapped: {len(res_mapped):,}")
print(f"Commuters mapped: {len(com_mapped):,}")
print(f"Merged total: {len(workers_merged):,}")

print("\nRows by population type:")
print(workers_merged["population_type"].value_counts())

print("\nShare works_in_brussels:")
print((workers_merged["works_in_brussels"].mean() * 100).round(2), "%")
print("\n")
print(workers_merged['works_in_brussels'].value_counts())